# 02 - 训练流水线

## 任务
递进训练3个方案（方案2 → 方案3 → 方案4）

## 策略
- **方案2 SFT**：使用 COT 数据微调 Qwen-0.5B
- **方案3 DPO**：使用偏好数据对齐（基于SFT模型）
- **方案4 GRPO**：组相对策略优化（基于DPO模型）

## 训练顺序
SFT → DPO → GRPO（后者继承前者权重）

## 运行环境
- **GPU（A10 24G / P100）**：正式训练
- **CPU**：仅验证流程可走通
- 配置自动适配，无需手动修改参数

In [ ]:
import os, sys

if os.path.exists('/kaggle'):
    WORK_DIR = '/kaggle/working'
elif os.path.exists('/mnt/workspace'):
    WORK_DIR = '/mnt/workspace'
else:
    WORK_DIR = os.getcwd()
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

print(f"工作目录: {os.getcwd()}")

In [ ]:
import subprocess, sys
for p in ['transformers', 'peft', 'accelerate', 'datasets', 'trl', 'tqdm']:
    try: __import__(p)
    except: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])
print('依赖就绪')

In [ ]:
from utils.pipeline_config import get_paths, get_device
from utils.common import enable_tf32, set_seed

paths = get_paths()
device = get_device()

if device != 'cpu':
    enable_tf32()
set_seed(42)

print(f"设备: {device}")
print(f"模型: {paths['base_model']}")
print(f"数据: {paths['data_dir']}")
print(f"输出: {paths['output_dir']}")
print(f"SFT数据: {paths['train_cot']}")
print(f"DPO数据: {paths['train_preference']}")

import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f}GB")

## ===== 方案2：SFT训练 =====

In [ ]:
from scheme2_data_enhancement.train_api import COTTrainer
from utils.pipeline_config import get_sft_config

sft_config = get_sft_config(device, paths)
print('方案2 SFT配置:')
for k, v in sft_config.items():
    print(f'  {k}: {v}')

In [ ]:
print('='*50)
print('开始方案2 SFT训练')
print('='*50)

sft_trainer = COTTrainer(sft_config)
sft_trainer.run(device=device)

SFT_PEFT_PATH = os.path.join(paths['output_dir'], 'scheme2_cot', 'final')
print(f'\n✓ 方案2 SFT训练完成！')
print(f'   模型路径: {SFT_PEFT_PATH}')

## ===== 方案3：DPO训练 =====

In [ ]:
from scheme3_dpo.train_dpo import train_dpo
from utils.pipeline_config import get_dpo_config

dpo_config = get_dpo_config(device, paths)
print('方案3 DPO配置:')
for k, v in dpo_config.items():
    print(f'  {k}: {v}')

In [ ]:
print('='*50)
print('开始方案3 DPO训练')
print('='*50)

dpo_trainer, dpo_model, dpo_tokenizer = train_dpo(**dpo_config)

DPO_MODEL_PATH = os.path.join(paths['output_dir'], 'scheme3_dpo', 'final')
print(f'\n✓ 方案3 DPO训练完成！')
print(f'   模型路径: {DPO_MODEL_PATH}')

## ===== 方案4：GRPO训练 =====

In [ ]:
from scheme4_grpo.train_grpo import train_grpo
from utils.pipeline_config import get_grpo_config

grpo_config = get_grpo_config(device, paths)
print('方案4 GRPO配置:')
for k, v in grpo_config.items():
    print(f'  {k}: {v}')

In [ ]:
print('='*50)
print('开始方案4 GRPO训练')
print('='*50)

grpo_trainer = train_grpo(**grpo_config)

GRPO_MODEL_PATH = os.path.join(paths['output_dir'], 'scheme4_grpo', 'final')
print(f'\n✓ 方案4 GRPO训练完成！')
print(f'   模型路径: {GRPO_MODEL_PATH}')

## ===== 训练完成总结 =====

In [ ]:
print('\n' + '='*50)
print('训练流水线完成')
print('='*50)

SCHEME_PATHS = [
    ('方案2 SFT', os.path.join(paths['output_dir'], 'scheme2_cot', 'final')),
    ('方案3 DPO', os.path.join(paths['output_dir'], 'scheme3_dpo', 'final')),
    ('方案4 GRPO', os.path.join(paths['output_dir'], 'scheme4_grpo', 'final')),
]

for name, path in SCHEME_PATHS:
    if os.path.exists(path):
        print(f'  ✓ {name}: {path}')
    else:
        print(f'  ✗ {name}: {path} (不存在)')